<a href="https://colab.research.google.com/github/Aswathi281099/Generative-Artificial-Intelligence/blob/main/AI_TASK_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**TASK 12**

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
class StudentVisionEncoder(nn.Module):
    """Simple, lightweight vision encoder for edge deployment."""
    def __init__(self, embed_dim=512):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, embed_dim)
        )

    def forward(self, x):
        return F.normalize(self.features(x), p=2, dim=-1)

In [8]:
class StudentTextEncoder(nn.Module):
    """Simple, lightweight text encoder for edge deployment."""
    def __init__(self, vocab_size=49408, embed_dim=512):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, 128)
        self.fc = nn.Linear(128, embed_dim)

    def forward(self, input_ids):
        x = self.token_embedding(input_ids).mean(dim=1)
        return F.normalize(self.fc(x), p=2, dim=-1)


class StudentCLIP(nn.Module):
    def __init__(self, embed_dim=512, logit_scale=20.0):
        super().__init__()
        self.vision_encoder = StudentVisionEncoder(embed_dim)
        self.text_encoder = StudentTextEncoder(vocab_size=49408, embed_dim=embed_dim)
        self.logit_scale = nn.Parameter(torch.tensor(logit_scale))

    def forward(self, pixel_values, input_ids):
        img_embeds = self.vision_encoder(pixel_values)
        text_embeds = self.text_encoder(input_ids)

        # Compute dynamic visual-text similarity logits
        logits_per_image = self.logit_scale * (img_embeds @ text_embeds.T)
        logits_per_text = logits_per_image.T
        return logits_per_image, logits_per_text, img_embeds, text_embeds

In [9]:
class DistillationLoss(nn.Module):
    """Combines Soft Logit KL-Divergence Loss and Feature Cosine Distance Loss."""
    def __init__(self, alpha_kl=0.5, alpha_cosine=0.5, temperature=2.0):
        super().__init__()
        self.alpha_kl = alpha_kl
        self.alpha_cosine = alpha_cosine
        self.temperature = temperature
        self.kl_div = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, student_embeds, teacher_embeds):
        # 1. Soft-label KL Divergence Loss on Logit Distributions
        p_student = F.log_softmax(student_logits / self.temperature, dim=-1)
        q_teacher = F.softmax(teacher_logits / self.temperature, dim=-1)
        loss_kl = self.kl_div(p_student, q_teacher) * (self.temperature ** 2)

        # 2. Cosine Distance Loss on Raw Embeddings (1 - Cosine Similarity)
        loss_cosine = (1.0 - F.cosine_similarity(student_embeds, teacher_embeds, dim=-1)).mean()

        # Weighted Composite Loss
        return self.alpha_kl * loss_kl + self.alpha_cosine * loss_cosine

In [10]:
teacher_name = "openai/clip-vit-base-patch32"
print(f"Loading frozen Teacher model ({teacher_name}) on {device}...")

processor = CLIPProcessor.from_pretrained(teacher_name)
teacher_model = CLIPModel.from_pretrained(teacher_name).to(device)

# Freeze static Teacher model parameters
for param in teacher_model.parameters():
    param.requires_grad = False
teacher_model.eval()

# Initialize uninitialized lightweight Student model
student_model = StudentCLIP(embed_dim=512).to(device)

# Composite loss module and optimizer
criterion = DistillationLoss(alpha_kl=0.5, alpha_cosine=0.5, temperature=2.0)
optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-3)

# Dummy Image-Text dataset inputs
raw_images = [torch.randint(0, 256, (3, 224, 224), dtype=torch.uint8) for _ in range(2)]
raw_texts = ["a photo of a cat", "a photo of a dog"]

inputs = processor(text=raw_texts, images=raw_images, return_tensors="pt", padding=True).to(device)

Loading frozen Teacher model (openai/clip-vit-base-patch32) on cpu...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [11]:
student_model.train()

# Forward pass through frozen Teacher
with torch.no_grad():
    teacher_outputs = teacher_model(**inputs)
    teacher_logits = teacher_outputs.logits_per_image
    teacher_img_embeds = teacher_outputs.image_embeds

# Forward pass through Student
s_logits_img, s_logits_text, s_img_embeds, s_text_embeds = student_model(
    inputs["pixel_values"], inputs["input_ids"]
)

# Calculate composite loss
loss = criterion(s_logits_img, teacher_logits, s_img_embeds, teacher_img_embeds)

# Optimization step
optimizer.zero_grad()
loss.backward()
optimizer.step()

print(f"Distillation Step Complete | Composite Loss: {loss.item():.4f}")

Distillation Step Complete | Composite Loss: 0.4880
